# Week 11 Lab — Transformers and Modern NLP Architectures

## Learning Goals
- Understand why sequential models (RNNs/LSTMs) fall short for long-range dependencies.
- Implement **scaled dot-product attention** from scratch.
- Implement **multi-head attention** and **sinusoidal positional encoding**.
- Assemble a **Transformer encoder block** with residuals and layer normalisation.
- Use pre-trained **BERT** to extract contextual embeddings.
- Fine-tune a transformer for **text classification** and analyse attention patterns.

## Part 0 — Setup

In [ ]:
!pip install -q torch transformers datasets scikit-learn matplotlib seaborn numpy pandas

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version :", torch.__version__)
print("All imports OK.")

---
## Part 1 — The Need for Transformers

Before the Transformer (Vaswani et al., 2017), NLP relied on **Recurrent Neural Networks** and **LSTMs**. These process tokens sequentially (left-to-right), which creates two bottlenecks:

1. **Sequential computation**: step t depends on step t−1, so you cannot parallelise across tokens.
2. **Long-range dependencies**: information must propagate through many time steps, causing **vanishing gradients**.

The Transformer eliminates both by computing attention over all tokens simultaneously in O(1) sequential steps.

| Property | RNN / LSTM | Transformer |
|----------|-----------|-------------|
| Computation order | Sequential | Fully parallel |
| Path between tokens i and j | O(n) steps | O(1) direct attention |
| Per-layer compute | O(n·d²) | O(n²·d) |
| Long-range dependency | Hard | Easy |

### Task 1A — Complexity Analysis

For a sequence of length n and model dimension d, fill in the table:

| Metric | Self-Attention | Recurrent (LSTM) |
|--------|---------------|------------------|
| Per-layer computation | O(n²·d) | O(n·d²) |
| Sequential operations | O(1) | O(n) |
| Maximum path length (i→j) | O(1) | O(n) |

**Question:** When is self-attention computationally cheaper than the recurrent layer? When is it more expensive? Give a concrete example using typical values (e.g. n=512, d=768).

_Your answer here._

---

### Task 1B — Long-Range Dependency

Sentence: *"The keys that the student who had studied abroad lost were found near the library entrance."*

1. Identify the long-range grammatical dependency.
2. How many recurrent steps must an LSTM bridge to connect subject and predicate?
3. How does self-attention resolve this in a single step?

_Your answer here._

---

### Discussion Questions

1. Why does the vanishing gradient problem particularly affect long-sequence RNN training? How does LSTM's gating partially fix it?
2. What is the **information bottleneck** in encoder-decoder RNNs? How did Bahdanau attention (2015) mitigate it?
3. What is **teacher forcing** and why is it unnecessary in Transformer training?

_Your answers here._

---
## Part 2 — Scaled Dot-Product Attention

The core operation of every Transformer layer:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- **Q** (queries), **K** (keys), **V** (values) are all linear projections of the input.
- The dot product $QK^\top$ measures compatibility between query and each key.
- Dividing by $\sqrt{d_k}$ prevents large magnitudes that push softmax into near-zero-gradient regions.
- The result is a weighted average of values, where weights reflect query-key alignment.

In [ ]:
# Coding Task A — Numerically stable softmax and scaled dot-product attention

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax along `axis`.
    Subtract the max before exponentiating to avoid overflow.

    Args:
        x    : input array of any shape
        axis : axis along which to apply softmax
    Returns:
        array of same shape with softmax applied
    """
    # TODO: subtract max for numerical stability, then compute exp / sum
    x_shifted = x - np.max(x, axis=axis, keepdims=True)  # TODO
    exp_x = np.exp(x_shifted)                            # TODO
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)  # TODO


def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: np.ndarray = None
) -> tuple:
    """
    Scaled dot-product attention.

    Args:
        Q    : (..., seq_q, d_k)
        K    : (..., seq_k, d_k)
        V    : (..., seq_k, d_v)
        mask : optional (..., seq_q, seq_k) additive mask (use -1e9 for positions to ignore)
    Returns:
        output : (..., seq_q, d_v) weighted value sum
        weights: (..., seq_q, seq_k) attention weights
    """
    d_k = Q.shape[-1]

    # TODO: compute dot product QK^T, scale by sqrt(d_k)
    scores = Q @ K.swapaxes(-2, -1) / math.sqrt(d_k)  # TODO

    # TODO: add mask if provided (mask contains -1e9 for forbidden positions)
    if mask is not None:
        scores = scores + mask  # TODO

    # TODO: softmax over the key dimension (last axis)
    weights = softmax(scores, axis=-1)  # TODO

    # TODO: weighted sum of values
    output = weights @ V  # TODO

    return output, weights


# Verify shapes
np.random.seed(0)
batch, seq, d_k, d_v = 2, 5, 8, 16
Q = np.random.randn(batch, seq, d_k)
K = np.random.randn(batch, seq, d_k)
V = np.random.randn(batch, seq, d_v)

out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output shape : {out.shape}   (expected {(batch, seq, d_v)})")
print(f"Weights shape: {w.shape}  (expected {(batch, seq, seq)})")
print(f"Weights sum (should be ~1.0): {w[0].sum(axis=-1)}")  # each row sums to 1

In [ ]:
# Coding Task B — Attention visualisation on a toy sequence

# Build a simple 6-token embedding by hand
# Tokens: ["the", "cat", "sat", "on", "the", "mat"]
np.random.seed(7)
tokens = ["the", "cat", "sat", "on", "the", "mat"]
n, d = len(tokens), 12

# Simulate Q=K=V from the same input (self-attention)
X = np.random.randn(1, n, d)      # batch=1
W_Q = np.random.randn(d, d)
W_K = np.random.randn(d, d)
W_V = np.random.randn(d, d)

Q_toy = X @ W_Q
K_toy = X @ W_K
V_toy = X @ W_V

_, attn_weights = scaled_dot_product_attention(Q_toy, K_toy, V_toy)

# TODO: plot the 6x6 attention weight matrix as a heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    attn_weights[0],        # shape (6, 6) — remove batch dim
    annot=True, fmt=".2f",
    xticklabels=tokens, yticklabels=tokens,
    cmap="Blues", ax=ax, vmin=0, vmax=1
)
ax.set_title("Self-Attention Weights (single head, random init)")
ax.set_xlabel("Key (attended-to)")
ax.set_ylabel("Query (attending)")
plt.tight_layout()
plt.show()
print("Observation: which token attends most to which?")

In [ ]:
# Coding Task C — Effect of temperature on attention sharpness

# Temperature τ replaces sqrt(d_k): scores /= τ
# Small τ → sharp (near-one-hot); large τ → flat (uniform)

def attention_with_temperature(Q, K, V, temperature):
    """Attention with configurable temperature instead of sqrt(d_k)."""
    # TODO: compute scores / temperature, then softmax, then @ V
    scores = Q @ K.swapaxes(-2, -1) / temperature   # TODO
    weights = softmax(scores, axis=-1)               # TODO
    output = weights @ V                             # TODO
    return output, weights


temperatures = [0.1, 1.0, 10.0]
labels = [f"τ = {t}" for t in temperatures]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, tau, label in zip(axes, temperatures, labels):
    _, w = attention_with_temperature(Q_toy, K_toy, V_toy, tau)
    sns.heatmap(
        w[0], annot=True, fmt=".2f",
        xticklabels=tokens, yticklabels=tokens,
        cmap="Blues", ax=ax, vmin=0, vmax=1,
        cbar=(ax == axes[-1])
    )
    ax.set_title(label)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query" if ax == axes[0] else "")

plt.suptitle("Attention Weight Sharpness vs Temperature", fontsize=13)
plt.tight_layout()
plt.show()

### Questions
1. What happens to attention weights when $d_k$ is large and the scaling factor is omitted? Why is this a training problem?
2. In **self-attention**, Q, K, V come from the same input. In **cross-attention** (decoder), Q comes from the decoder and K, V from the encoder. What does each mechanism achieve?
3. What is a **causal mask** and why is it essential for autoregressive language models?
4. From the temperature experiment: describe the practical risk of τ → 0 (very sharp) and τ → ∞ (very flat) during training.

_Your answers here._

---
## Part 3 — Multi-Head Attention and Positional Encoding

**Multi-head attention** runs h parallel attention heads, each with its own learned projections:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W^O$$
$$\text{head}_i = \text{Attention}(QW_Q^i,\; KW_K^i,\; VW_V^i)$$

Different heads can specialise in different relationships (syntax, coreference, local/global).

Because the Transformer has no recurrence, it needs explicit **positional encoding** added to each token's embedding:

$$PE_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) \qquad PE_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
# Coding Task A — Multi-Head Attention (PyTorch)

class MultiHeadAttention(nn.Module):
    """
    Multi-head scaled dot-product attention.

    Args:
        d_model   : model embedding dimension
        num_heads : number of attention heads (d_model must be divisible by num_heads)
    """
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # TODO: define four linear projections: W_Q, W_K, W_V, W_O
        # Each is nn.Linear(d_model, d_model, bias=False)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # TODO
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # TODO
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # TODO
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # TODO

    def split_heads(self, x: torch.Tensor, batch_size: int) -> torch.Tensor:
        """
        Reshape (batch, seq, d_model) → (batch, num_heads, seq, d_k).
        """
        # TODO:
        # 1. reshape x to (batch, seq, num_heads, d_k)
        # 2. transpose to (batch, num_heads, seq, d_k)
        x = x.view(batch_size, -1, self.num_heads, self.d_k)  # TODO
        return x.transpose(1, 2)                               # TODO

    def forward(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Args:
            Q, K, V : (batch, seq, d_model)
            mask    : optional (batch, 1, seq, seq) boolean mask (True = ignore)
        Returns:
            output  : (batch, seq, d_model)
        """
        batch_size = Q.size(0)

        # TODO: project Q, K, V with learned weight matrices
        Q = self.W_Q(Q)  # TODO
        K = self.W_K(K)  # TODO
        V = self.W_V(V)  # TODO

        # TODO: split into multiple heads
        Q = self.split_heads(Q, batch_size)  # (batch, heads, seq, d_k)  # TODO
        K = self.split_heads(K, batch_size)  # TODO
        V = self.split_heads(V, batch_size)  # TODO

        # TODO: scaled dot-product attention (use PyTorch ops)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # TODO
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))      # TODO
        attn_weights = F.softmax(scores, dim=-1)                 # TODO
        context = attn_weights @ V                               # TODO

        # TODO: concatenate heads: (batch, heads, seq, d_k) → (batch, seq, d_model)
        context = context.transpose(1, 2).contiguous()               # TODO
        context = context.view(batch_size, -1, self.d_model)         # TODO

        # TODO: project output
        output = self.W_O(context)  # TODO
        return output


# Verify
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)   # batch=2, seq=10, d_model=64
out = mha(x, x, x)
print(f"MultiHeadAttention output shape: {out.shape}  (expected torch.Size([2, 10, 64]))")

In [ ]:
# Coding Task B — Sinusoidal Positional Encoding

def positional_encoding(max_len: int, d_model: int) -> np.ndarray:
    """
    Compute the sinusoidal positional encoding matrix.

    PE[pos, 2i]   = sin(pos / 10000^(2i / d_model))
    PE[pos, 2i+1] = cos(pos / 10000^(2i / d_model))

    Args:
        max_len : maximum sequence length
        d_model : embedding dimension
    Returns:
        pe : (max_len, d_model) numpy array
    """
    # TODO:
    # 1. Create position array shape (max_len, 1)
    # 2. Create dimension index array shape (1, d_model // 2)
    # 3. Compute div_term = 10000^(2i/d_model)
    # 4. Fill even columns with sin, odd columns with cos
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]                          # TODO
    i = np.arange(d_model // 2)[np.newaxis, :]                            # TODO
    div_term = np.power(10000.0, 2 * i / d_model)                        # TODO
    pe[:, 0::2] = np.sin(position / div_term)                            # TODO: even dims
    pe[:, 1::2] = np.cos(position / div_term)                            # TODO: odd dims
    return pe


# Verify shape
pe = positional_encoding(max_len=50, d_model=64)
print(f"Positional encoding shape: {pe.shape}  (expected (50, 64))")
print(f"Value range: [{pe.min():.3f}, {pe.max():.3f}]  (expected [-1, 1])")

In [ ]:
# Coding Task C — Visualise positional encodings

pe_viz = positional_encoding(max_len=60, d_model=128)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PE matrix heatmap
ax = axes[0]
im = ax.imshow(pe_viz, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Position")
ax.set_title("Sinusoidal Positional Encoding Matrix")
plt.colorbar(im, ax=ax)

# Right: dot-product similarity PE @ PE^T (should show locality)
ax = axes[1]
# TODO: compute similarity matrix = pe_viz @ pe_viz.T, normalised by d_model
similarity = pe_viz @ pe_viz.T / pe_viz.shape[1]   # TODO
im2 = ax.imshow(similarity, aspect='auto', cmap='Blues')
ax.set_xlabel("Position j")
ax.set_ylabel("Position i")
ax.set_title("Position Similarity (PE·PEᵀ / d_model)")
plt.colorbar(im2, ax=ax)

plt.tight_layout()
plt.savefig("positional_encoding.png", dpi=100)
plt.show()
print("Saved: positional_encoding.png")

### Questions
1. Why does multi-head attention learn richer representations than single-head attention of the same total dimension?
2. The original Transformer uses **fixed sinusoidal** encodings. BERT and GPT use **learned positional embeddings**. What are the trade-offs (length generalisation, flexibility, compute)?
3. From the similarity plot: what does it tell you about how positions relate to each other under sinusoidal encoding?
4. What is **Rotary Position Embedding (RoPE)** used in LLaMA/GPT-Neo? Give a high-level intuition.

_Your answers here._

---
## Part 4 — The Transformer Encoder Block

One Transformer encoder layer:
```
x = LayerNorm(x + MultiHeadAttention(x, x, x))   # self-attention sublayer
x = LayerNorm(x + FFN(x))                         # feed-forward sublayer
```

The **Feed-Forward Network (FFN)** is a 2-layer MLP applied independently to every token position:

$$\text{FFN}(x) = \max(0,\; xW_1 + b_1)\,W_2 + b_2$$

with inner dimension $d_{ff} = 4 \times d_{\text{model}}$ (e.g. 512 → 2048 → 512).

In [ ]:
# Coding Task A — Position-wise Feed-Forward Network

class PositionwiseFFN(nn.Module):
    """
    Two-layer FFN applied identically to every token position.

    FFN(x) = ReLU(xW1 + b1) W2 + b2

    Args:
        d_model : input and output dimension
        d_ff    : inner (hidden) dimension (typically 4 * d_model)
        dropout : dropout rate applied after the first linear layer
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: define two linear layers and a dropout + relu
        self.linear1  = nn.Linear(d_model, d_ff)    # TODO
        self.linear2  = nn.Linear(d_ff, d_model)    # TODO
        self.dropout  = nn.Dropout(dropout)          # TODO

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch, seq, d_model)
        Returns:
            (batch, seq, d_model)
        """
        # TODO: apply linear1 → relu → dropout → linear2
        x = F.relu(self.linear1(x))  # TODO
        x = self.dropout(x)           # TODO
        x = self.linear2(x)           # TODO
        return x


# Quick test
ffn = PositionwiseFFN(d_model=64, d_ff=256)
x_test = torch.randn(2, 10, 64)
print(f"FFN output shape: {ffn(x_test).shape}  (expected torch.Size([2, 10, 64]))")

In [ ]:
# Coding Task B — Transformer Encoder Layer

class TransformerEncoderLayer(nn.Module):
    """
    Single Transformer encoder block:
        x = LayerNorm(x + MHA(x, x, x))
        x = LayerNorm(x + FFN(x))

    Args:
        d_model   : model dimension
        num_heads : number of attention heads
        d_ff      : feed-forward inner dimension
        dropout   : dropout rate
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: instantiate sub-modules
        self.self_attn = MultiHeadAttention(d_model, num_heads)  # TODO
        self.ffn       = PositionwiseFFN(d_model, d_ff, dropout)  # TODO
        self.norm1     = nn.LayerNorm(d_model)                    # TODO
        self.norm2     = nn.LayerNorm(d_model)                    # TODO
        self.dropout   = nn.Dropout(dropout)                      # TODO

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x    : (batch, seq, d_model)
            mask : optional padding mask
        Returns:
            (batch, seq, d_model)
        """
        # TODO: sublayer 1 — self-attention with residual and layer norm
        attn_out = self.self_attn(x, x, x, mask)         # TODO
        x = self.norm1(x + self.dropout(attn_out))       # TODO

        # TODO: sublayer 2 — FFN with residual and layer norm
        ffn_out = self.ffn(x)                            # TODO
        x = self.norm2(x + self.dropout(ffn_out))        # TODO

        return x


# Verify
layer = TransformerEncoderLayer(d_model=64, num_heads=8, d_ff=256)
x_in = torch.randn(2, 10, 64)
x_out = layer(x_in)
print(f"Encoder layer output shape: {x_out.shape}  (expected torch.Size([2, 10, 64]))")

In [ ]:
# Coding Task C — Full Encoder: stack layers and count parameters

class TransformerEncoder(nn.Module):
    """
    Stack of N TransformerEncoderLayer blocks.

    Args:
        num_layers : number of stacked layers
        d_model, num_heads, d_ff, dropout: passed to each layer
    """
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # TODO: create a ModuleList of num_layers TransformerEncoderLayer
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])  # TODO
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # TODO: pass x through each layer sequentially
        for layer in self.layers:  # TODO
            x = layer(x, mask)     # TODO
        return self.norm(x)


# Build a 2-layer encoder
encoder = TransformerEncoder(num_layers=2, d_model=64, num_heads=8, d_ff=256)
x_in = torch.randn(2, 10, 64)
x_out = encoder(x_in)
print(f"Encoder (2 layers) output shape: {x_out.shape}")

# Count parameters
total_params = sum(p.numel() for p in encoder.parameters())
print(f"Total trainable parameters: {total_params:,}")

In [ ]:
# Coding Task D — Compare with PyTorch built-in

# nn.TransformerEncoderLayer uses the same architecture
pytorch_layer = nn.TransformerEncoderLayer(
    d_model=64, nhead=8, dim_feedforward=256,
    dropout=0.1, batch_first=True
)
pytorch_encoder = nn.TransformerEncoder(pytorch_layer, num_layers=2)

x_in = torch.randn(2, 10, 64)
x_out_pt = pytorch_encoder(x_in)
print(f"PyTorch encoder output shape: {x_out_pt.shape}")

pt_params = sum(p.numel() for p in pytorch_encoder.parameters())
print(f"PyTorch encoder parameters : {pt_params:,}")
print(f"Our encoder parameters     : {total_params:,}")
print()
print("Parameter counts may differ slightly due to bias terms and implementation details.")

### Questions
1. Why are **residual connections** critical for training deep Transformers? What problem do they solve?
2. How does **layer normalisation** differ from batch normalisation? Why is LayerNorm preferred in NLP Transformers?
3. The FFN inner dimension is typically 4× the model dimension. What is the effect of increasing this ratio? Which model family uses a larger ratio (e.g. 8× or more)?
4. Why does applying the FFN independently to each position still allow the model to combine cross-token information?

_Your answers here._

---
## Part 5 — Pre-trained Transformers with HuggingFace

**BERT** (Devlin et al., 2018) is pre-trained on BookCorpus + Wikipedia with:
1. **Masked Language Modeling (MLM)**: 15% of tokens masked; the model predicts them.
2. **Next Sentence Prediction (NSP)**: the model classifies whether two sentences are consecutive.

This yields **contextual embeddings**: the word *"bank"* has different vector representations in *"river bank"* vs *"bank account"*.

In [ ]:
# Coding Task A — Load BERT and extract embeddings

from transformers import BertTokenizer, BertModel
import torch

MODEL_NAME = "bert-base-uncased"
tokenizer_bert = BertTokenizer.from_pretrained(MODEL_NAME)
model_bert = BertModel.from_pretrained(MODEL_NAME)
model_bert.eval()

print(f"BERT loaded: {MODEL_NAME}")
print(f"Parameters : {sum(p.numel() for p in model_bert.parameters()):,}")

# Encode three sentences
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Attention mechanisms revolutionised natural language processing.",
    "She deposited money at the bank.",
]

# TODO: tokenise with return_tensors='pt', padding=True, truncation=True
inputs = tokenizer_bert(
    sentences, return_tensors="pt", padding=True, truncation=True
)  # TODO

with torch.no_grad():
    # TODO: run through BERT, get last_hidden_state and pooler_output
    outputs = model_bert(**inputs)  # TODO

last_hidden  = outputs.last_hidden_state   # (batch, seq, 768)
cls_embeddings = last_hidden[:, 0, :]     # [CLS] token

print(f"\nlast_hidden_state shape : {last_hidden.shape}")
print(f"[CLS] embeddings shape  : {cls_embeddings.shape}")

# Cosine similarity between [CLS] embeddings
from torch.nn.functional import cosine_similarity
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(cls_embeddings[i:i+1], cls_embeddings[j:j+1]).item()
        print(f"  sim(sent {i+1}, sent {j+1}) = {sim:.4f}")

In [ ]:
# Coding Task B — Contextual vs static embeddings: the word "bank"

bank_sentences = [
    "She sat on the bank of the river and watched the water.",
    "He deposited all his savings at the bank on Monday.",
    "The data bank was encrypted with a 256-bit key.",
]

# TODO: tokenise each sentence individually, find position of 'bank', extract its embedding
def get_word_embedding(sentence, word, tokenizer, model):
    """
    Return the last-hidden-state vector for the first occurrence of `word` in `sentence`.
    Returns None if word is not found.
    """
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Find token index for `word` (may be split into sub-words; take first sub-word)
    # TODO: find index where token starts with word (case-insensitive)
    target_idx = None
    for idx, tok in enumerate(tokens):
        if tok.lower() == word.lower() or tok.lower() == "##" + word.lower():
            target_idx = idx
            break
        # handle sub-word: the word 'bank' → 'bank' directly (no ##)
        if word.lower() in tok.lower() and not tok.startswith("##"):
            target_idx = idx
            break

    if target_idx is None:
        return None, tokens

    with torch.no_grad():
        outputs = model(**inputs)  # TODO

    # TODO: return embedding at target_idx
    embedding = outputs.last_hidden_state[0, target_idx, :]  # TODO
    return embedding.numpy(), tokens


bank_embeddings = []
for sent in bank_sentences:
    emb, toks = get_word_embedding(sent, "bank", tokenizer_bert, model_bert)
    bank_embeddings.append(emb)
    print(f"Tokens: {toks}")
    print(f"  'bank' embedding norm: {np.linalg.norm(emb):.4f}")
    print()

# TODO: compute pairwise cosine similarities between the three 'bank' embeddings
print("Pairwise cosine similarities for 'bank' in different contexts:")
for i in range(3):
    for j in range(i+1, 3):
        e_i = torch.tensor(bank_embeddings[i]).unsqueeze(0)
        e_j = torch.tensor(bank_embeddings[j]).unsqueeze(0)
        sim = cosine_similarity(e_i, e_j).item()  # TODO
        print(f"  Context {i+1} vs {j+1}: {sim:.4f}")

print("\nWith static embeddings all three 'bank' vectors would be identical (similarity = 1.0).")

In [ ]:
# Coding Task C — Masked Language Modeling with BERT fill-mask pipeline

from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")

# Five masked sentences covering different linguistic phenomena
masked_sentences = [
    # Grammatical agreement
    "The dogs [MASK] loudly every morning.",
    # World knowledge
    "The capital of France is [MASK].",
    # Analogy / semantic field
    "The doctor examined the patient and the [MASK] took notes.",
    # Idiom / colocation
    "It's raining cats and [MASK] outside.",
    # Sentiment / sentiment-aware prediction
    "The movie was absolutely [MASK]. I loved every minute of it.",
]

print("BERT Masked Language Model Predictions:\n")
for sentence in masked_sentences:
    results = fill_mask(sentence)
    # TODO: print top-3 predictions with scores
    print(f"Input : {sentence}")
    for r in results[:3]:  # TODO
        print(f"  [{r['score']:.4f}] {r['token_str']}")
    print()

### Questions
1. What is the difference between **bidirectional** (BERT) and **autoregressive** (GPT) pre-training? Which suits classification? Which suits generation?
2. Why is the **[CLS] token** embedding used for sequence-level tasks? What does it learn during MLM pre-training?
3. What is **sub-word tokenisation** (WordPiece / BPE)? Why is it preferred over word-level or character-level tokenisation?
4. From Task 5B: did BERT successfully distinguish the different senses of *"bank"*? Which two contexts were most similar and why?

_Your answers here._

---
## Part 6 — Fine-tuning BERT for Text Classification

**Fine-tuning** attaches a small task-specific head (a linear layer on the [CLS] embedding) and trains on labeled data. Because BERT's representations are already rich, even 200 examples can yield strong performance.

In [ ]:
# Coding Task A — Zero-shot baseline with a fine-tuned pipeline

from datasets import load_dataset
from transformers import pipeline

# Load SST-2 validation split (50 samples)
dataset = load_dataset("glue", "sst2", split="validation[:50]")
texts  = dataset["sentence"]
labels = dataset["label"]   # 0 = negative, 1 = positive

# Pre-trained DistilBERT fine-tuned on SST-2 (used as the "oracle" zero-shot baseline)
sentiment_pipeline = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True
)

# TODO: run inference on all 50 texts and compute accuracy
predictions = sentiment_pipeline(texts)
pred_labels = [1 if p["label"] == "POSITIVE" else 0 for p in predictions]  # TODO
acc_zero_shot = accuracy_score(labels, pred_labels)                          # TODO

print(f"Zero-shot (pre-fine-tuned SST-2) accuracy on 50 samples: {acc_zero_shot:.4f}")

In [ ]:
# Coding Task B — Manual fine-tuning loop on 200 SST-2 training examples

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Load data
train_dataset = load_dataset("glue", "sst2", split="train[:200]")
val_dataset   = load_dataset("glue", "sst2", split="validation[:50]")

MODEL_ID = "distilbert-base-uncased"
tokenizer_clf = DistilBertTokenizer.from_pretrained(MODEL_ID)


class SST2Dataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.texts  = data["sentence"]
        self.labels = data["label"]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # TODO: tokenise self.texts[idx] with padding and truncation
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )  # TODO
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_loader = DataLoader(SST2Dataset(train_dataset, tokenizer_clf), batch_size=16, shuffle=True)
val_loader   = DataLoader(SST2Dataset(val_dataset,   tokenizer_clf), batch_size=16)

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_clf = DistilBertForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model_clf = model_clf.to(device)

# TODO: AdamW optimiser with lr=2e-5
optimizer = AdamW(model_clf.parameters(), lr=2e-5)  # TODO


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch   = batch["label"].to(device)
            # TODO: forward pass, get logits
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # TODO
            preds = outputs.logits.argmax(dim=-1)                                # TODO
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())
    return accuracy_score(all_labels, all_preds)


# Training loop — 3 epochs
for epoch in range(3):
    model_clf.train()
    total_loss = 0
    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch   = batch["label"].to(device)

        # TODO: forward pass (pass labels for automatic loss computation)
        optimizer.zero_grad()                                                        # TODO
        outputs = model_clf(                                                         # TODO
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        loss = outputs.loss                                                          # TODO
        loss.backward()                                                              # TODO
        optimizer.step()                                                             # TODO
        total_loss += loss.item()

    val_acc = evaluate(model_clf, val_loader, device)
    print(f"Epoch {epoch+1}/3 | loss={total_loss/len(train_loader):.4f} | val_acc={val_acc:.4f}")

print(f"\nFine-tuned accuracy (200 examples, 3 epochs): {val_acc:.4f}")
print(f"Zero-shot baseline accuracy                  : {acc_zero_shot:.4f}")

In [ ]:
# Coding Task C — Learning curve: accuracy vs training set size

train_sizes = [50, 100, 200, 400]
val_accuracies = []

for n_train in train_sizes:
    # Load n_train samples
    subset = load_dataset("glue", "sst2", split=f"train[:{n_train}]")
    loader = DataLoader(SST2Dataset(subset, tokenizer_clf), batch_size=16, shuffle=True)

    # Fresh model for each size
    m = DistilBertForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2).to(device)
    opt = AdamW(m.parameters(), lr=2e-5)

    # TODO: train for 3 epochs, evaluate on val_loader
    for _ in range(3):
        m.train()
        for batch in loader:
            opt.zero_grad()
            out = m(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["label"].to(device)
            )  # TODO
            out.loss.backward()    # TODO
            opt.step()             # TODO

    acc = evaluate(m, val_loader, device)  # TODO
    val_accuracies.append(acc)
    print(f"  n_train={n_train:4d}  val_acc={acc:.4f}")

# TODO: plot learning curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_sizes, val_accuracies, marker='o', color='steelblue')  # TODO
ax.set_xlabel("Training set size")
ax.set_ylabel("Validation Accuracy")
ax.set_title("Fine-tuning Learning Curve (DistilBERT on SST-2)")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("finetune_learning_curve.png", dpi=100)
plt.show()
print("Saved: finetune_learning_curve.png")

In [ ]:
# Coding Task D — Confusion matrix

model_clf.eval()
all_preds_cm, all_labels_cm = [], []
with torch.no_grad():
    for batch in val_loader:
        out = model_clf(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        preds = out.logits.argmax(dim=-1).cpu().numpy()
        all_preds_cm.extend(preds)
        all_labels_cm.extend(batch["label"].numpy())

# TODO: compute and plot confusion matrix
cm = confusion_matrix(all_labels_cm, all_preds_cm)  # TODO

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — DistilBERT SST-2 Fine-tuning")
plt.tight_layout()
plt.show()

### Questions
1. Why does fine-tuning DistilBERT on only 200 examples still give reasonable accuracy? What does this reveal about pre-training?
2. What is **catastrophic forgetting** in fine-tuning? How does the low learning rate (2e-5) mitigate it?
3. Describe **LoRA** (Low-Rank Adaptation) as an alternative to full fine-tuning. What is its memory advantage?
4. From the confusion matrix: which class is harder to classify and why might that be?

_Your answers here._

---
## Part 7 — Attention Visualisation and Why Transformers Transformed NLP

Attention weights are one of the few intrinsic interpretability tools in deep learning. Different heads in different layers often specialise: some aggregate global context into [CLS], some track local bigrams, some align subject to verb.

In [ ]:
# Coding Task A — BERT attention head visualisation

from transformers import BertModel, BertTokenizer

bert_attn = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_attn.eval()

sentence = "The cat sat on the mat ."
inputs = tokenizer_bert(sentence, return_tensors="pt")
tokens_list = tokenizer_bert.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = bert_attn(**inputs)

# outputs.attentions: tuple of (batch, heads, seq, seq) per layer
# TODO: extract layer 5 (0-indexed), all 12 heads
layer_idx = 5
attn_layer5 = outputs.attentions[layer_idx][0]  # (12, seq, seq)  — remove batch dim

# Visualise 4 heads
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
head_indices = [0, 1, 4, 7]   # pick 4 varied heads

for ax, h in zip(axes.flatten(), head_indices):
    # TODO: extract head h attention matrix and plot as heatmap
    w = attn_layer5[h].numpy()   # (seq, seq)  # TODO
    sns.heatmap(
        w, annot=True, fmt=".2f",
        xticklabels=tokens_list,
        yticklabels=tokens_list,
        cmap="Blues", ax=ax, vmin=0,
        cbar=False
    )
    ax.set_title(f"Layer {layer_idx+1}, Head {h+1}")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.suptitle("BERT Attention Weights — Layer 6 (bert-base-uncased)", fontsize=13)
plt.tight_layout()
plt.show()

print("\nDescribe what each head appears to attend to:")
print("  Head 1: ?")
print("  Head 2: ?")
print("  Head 5: ?")
print("  Head 8: ?")

In [ ]:
# Coding Task B — Subject-verb attention

# Find which (layer, head) most strongly attends from subject to verb
test_sentences_sv = [
    ("The dogs bark loudly .",            "dogs", "bark"),
    ("The student who won the award studies biology .", "student", "studies"),
    ("Scientists published their results last week .",  "scientists", "published"),
]

print("Subject → Verb Attention Analysis")
print("=" * 60)

for sentence, subj, verb in test_sentences_sv:
    inputs_sv = tokenizer_bert(sentence, return_tensors="pt")
    toks = tokenizer_bert.convert_ids_to_tokens(inputs_sv["input_ids"][0])

    with torch.no_grad():
        outs_sv = bert_attn(**inputs_sv)

    # TODO: find token indices for subject and verb
    subj_idx = next((i for i, t in enumerate(toks) if t.lower() == subj.lower()), None)  # TODO
    verb_idx = next((i for i, t in enumerate(toks) if t.lower() == verb.lower()), None)  # TODO

    if subj_idx is None or verb_idx is None:
        print(f"  Could not find '{subj}' or '{verb}' in {toks}")
        continue

    best_layer, best_head, best_weight = -1, -1, -1
    for layer_i, layer_attn in enumerate(outs_sv.attentions):
        # layer_attn: (batch, heads, seq, seq)
        for head_i in range(layer_attn.size(1)):
            # TODO: get attention weight from subj_idx to verb_idx
            weight = layer_attn[0, head_i, subj_idx, verb_idx].item()  # TODO
            if weight > best_weight:
                best_weight = weight
                best_layer, best_head = layer_i, head_i

    print(f"Sentence : {sentence}")
    print(f"  Subject='{subj}' (idx {subj_idx}) → Verb='{verb}' (idx {verb_idx})")
    print(f"  Best: Layer {best_layer+1}, Head {best_head+1}, weight={best_weight:.4f}")
    print()

### The Transformer Revolution

| Year | Milestone | Impact |
|------|-----------|--------|
| 2017 | Transformer (Vaswani et al.) | Replaces RNNs for MT; 25× training speedup |
| 2018 | BERT (Devlin et al.) | Pre-train → fine-tune; SOTA on 11 NLP tasks overnight |
| 2018 | GPT-1 (Radford et al.) | Autoregressive pre-training for generation |
| 2019 | GPT-2; RoBERTa; XLNet | Scale and training strategy matter enormously |
| 2020 | GPT-3 (175B params) | Few-shot prompting; in-context learning emerges |
| 2021 | CLIP; DALL-E | Transformers conquer vision and multimodality |
| 2022 | ChatGPT; Instruction tuning | RLHF aligns generation to human preferences |
| 2023–24 | GPT-4; Gemini; LLaMA; Claude | Reasoning, coding, multimodal at scale |

### Discussion Questions

1. The Transformer's success rests on three pillars: **parallelism**, **pre-training**, and **scale**. Explain each and how they reinforce each other.
2. What are **scaling laws** (Kaplan et al., 2020)? How do they allow researchers to predict model performance from compute budget alone?
3. Why did BERT's release in 2018 immediately make many task-specific architectures (hand-tuned BiLSTM + CRF for NER, etc.) obsolete?
4. Transformers have been applied to images (ViT), audio (Whisper), protein structures (AlphaFold 2), and code (Codex). What common property of these domains enables the same architecture to work?
5. Name at least **three fundamental limitations** of current Transformer-based LLMs (e.g. quadratic attention, hallucination, reasoning).

_Your answers here._

---
## Submission

- Complete all `TODO` sections in the code cells.
- Answer all written questions in the markdown cells.
- Include in your submission:
  - The **attention heatmap** from Part 2B
  - The **temperature comparison plot** from Part 2C
  - The **positional encoding visualisation** from Part 3C (`positional_encoding.png`)
  - The **fine-tuning learning curve** from Part 6C (`finetune_learning_curve.png`)
  - The **confusion matrix** from Part 6D
  - The **BERT attention head visualisations** from Part 7A
  - Your **attention pattern table** from Part 7B
- Submit your completed `.ipynb` file as instructed.